# NumLab — Numerical & Stats Toolkit

A visual walkthrough of the toolkit in [`src/toolkit.py`](../src/toolkit.py).

The running theme: **express the operation on whole arrays**. Every section below
shows a manual, loop-based version next to the vectorized one, and checks that they
agree before showing how much faster the vectorized form is.

1. Array creation & distributions
2. Descriptive statistics — manual vs NumPy
3. Vectorized operations & broadcasting
4. Linear algebra
5. Simulation — Monte Carlo estimate of pi
6. Performance — loops vs vectorization
7. Saving and loading arrays

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import toolkit as tk

SEED = 42
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('numpy', np.__version__)

## 1. Array creation & distributions

Every random helper takes a `seed`, so nothing in this notebook changes between runs.

In [ ]:
print('arange(0, 10, 2) ', tk.arange(0, 10, 2))
print('linspace(0, 1, 5)', tk.linspace(0, 1, 5))
print('identity(3)      ', tk.identity(3).tolist())
print('random_normal(5) ', np.round(tk.random_normal(5, seed=SEED), 3))

same = np.array_equal(tk.random_normal(5, seed=SEED), tk.random_normal(5, seed=SEED))
print('\nsame seed -> same numbers:', same)

In [ ]:
specs = [
    ('normal', {'loc': 0.0, 'scale': 1.0}),
    ('uniform', {'low': -2.0, 'high': 2.0}),
    ('exponential', {'scale': 1.0}),
    ('poisson', {'lam': 3.0}),
    ('binomial', {'n': 20, 'p': 0.3}),
    ('lognormal', {'mean': 0.0, 'sigma': 0.6}),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, params) in zip(axes.ravel(), specs):
    sample = tk.sample_distribution(name, 20_000, seed=SEED, **params)
    ax.hist(sample, bins=40, color='steelblue', edgecolor='none')
    ax.set_title(f'{name}  (mean={sample.mean():.2f})')
fig.suptitle('sample_distribution — 20,000 draws each')
fig.tight_layout()
plt.show()

## 2. Descriptive statistics — manual vs NumPy

Each statistic is implemented twice: once as an explicit Python loop (`*_manual`) and
once as the NumPy call. They should agree to floating-point noise (~1e-15).

In [ ]:
data = tk.random_normal(5_000, loc=10.0, scale=2.5, seed=SEED)

rows = [
    ('mean', tk.mean_manual(data), float(tk.mean(data))),
    ('median', tk.median_manual(data), float(tk.median(data))),
    ('variance', tk.variance_manual(data, ddof=1), float(tk.variance(data, ddof=1))),
    ('std', tk.std_manual(data, ddof=1), float(tk.std(data, ddof=1))),
    ('p25', tk.percentile_manual(data, 25), float(tk.percentile(data, 25))),
    ('p75', tk.percentile_manual(data, 75), float(tk.percentile(data, 75))),
]

print(f"{'statistic':<10}{'manual':>14}{'numpy':>14}{'|diff|':>12}")
print('-' * 50)
for label, manual, vectorized in rows:
    print(f'{label:<10}{manual:>14.8f}{vectorized:>14.8f}{abs(manual - vectorized):>12.1e}')

In [ ]:
summary = tk.describe(data, ddof=1)
print(summary)

fig, ax = plt.subplots()
ax.hist(data, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(summary.mean, color='crimson', lw=2, label=f'mean {summary.mean:.2f}')
ax.axvline(summary.median, color='orange', lw=2, ls='--', label=f'median {summary.median:.2f}')
ax.axvspan(summary.q1, summary.q3, color='crimson', alpha=0.1, label=f'IQR {summary.iqr:.2f}')
ax.set_title('describe() on 5,000 normal samples')
ax.legend()
plt.show()

## 3. Vectorized operations & broadcasting

`zscore`, `minmax_scale` and `moving_average` all operate on the whole array at once.
`pairwise_distances` is the clearest broadcasting example: reshaping to `(n, 1, d)` and
`(1, m, d)` produces the full `(n, m)` distance matrix without a Python loop.

In [ ]:
t = tk.linspace(0, 4 * np.pi, 300)
signal = np.sin(t) + tk.random_normal(300, scale=0.35, seed=SEED)
window = 21
smoothed = tk.moving_average(signal, window)

fig, ax = plt.subplots()
ax.plot(t, signal, color='lightsteelblue', label='raw signal')
ax.plot(t[window - 1:], smoothed, color='crimson', lw=2, label=f'moving_average(window={window})')
ax.plot(t, np.sin(t), color='black', ls='--', lw=1, label='true sin(t)')
ax.set_title('Smoothing a noisy signal')
ax.legend()
plt.show()

In [ ]:
raw = tk.random_normal(400, loc=50.0, scale=12.0, seed=SEED)
standardised = tk.zscore(raw)
scaled = tk.minmax_scale(raw)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, values, title in zip(
    axes,
    [raw, standardised, scaled],
    [f'raw (mean={raw.mean():.1f})', f'zscore (mean={standardised.mean():.1e}, std={standardised.std():.2f})', 'minmax_scale to [0, 1]'],
):
    ax.hist(values, bins=30, color='steelblue', edgecolor='none')
    ax.set_title(title, fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# Three well-separated clusters -> a block-structured distance matrix.
generator = tk.rng_from(SEED)
centres = np.array([[0.0, 0.0], [6.0, 0.0], [3.0, 5.0]])
points = np.vstack([centre + tk.random_normal((8, 2), scale=0.6, seed=generator) for centre in centres])

distances = tk.pairwise_distances(points)
print('points', points.shape, '-> distances', distances.shape)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.5))
left.scatter(points[:, 0], points[:, 1], c=np.repeat([0, 1, 2], 8), cmap='viridis', s=45)
left.set_title('24 points, 3 clusters')
left.set_aspect('equal')
image = right.imshow(distances, cmap='magma')
right.set_title('pairwise_distances (broadcasting)')
right.grid(False)
fig.colorbar(image, ax=right, label='euclidean distance')
fig.tight_layout()
plt.show()

In [ ]:
contaminated = np.append(tk.random_normal(500, loc=20.0, scale=3.0, seed=SEED), [80.0, -30.0, 75.0])
mask = tk.outlier_mask(contaminated, threshold=3.0)
cleaned = tk.drop_outliers(contaminated, threshold=3.0)

print(f'{contaminated.size} values in, {cleaned.size} kept, {mask.sum()} flagged at |z| > 3')
print('flagged values:', contaminated[mask])
print(f'std before {contaminated.std():.2f} -> after {cleaned.std():.2f}')

## 4. Linear algebra

`matmul_manual` is the textbook triple loop; `matmul` is `@`, which dispatches to BLAS.
Same answer, very different speed (measured in section 6).

In [ ]:
generator = tk.rng_from(SEED)
A = tk.random_normal((4, 4), seed=generator)
B = tk.random_normal((4, 4), seed=generator)

print('manual == vectorized :', np.allclose(tk.matmul_manual(A, B), tk.matmul(A, B)))
print('trace(A)             :', round(tk.trace(A), 4))
print('det(A)               :', round(tk.determinant(A), 4))
print('A @ inv(A) == I      :', np.allclose(tk.matmul(A, tk.inverse(A)), tk.identity(4)))

b = np.ones(4)
x = tk.solve(A, b)
print('\nsolve(A, b)          :', np.round(x, 4))
print('residual |Ax - b|    : {:.2e}'.format(tk.norm(A @ x - b)))

values, vectors = tk.eigen(A)
print('eigenvalues          :', np.round(values, 3))

In [ ]:
# Vector geometry: dot product, angle and projection.
u = np.array([4.0, 1.0])
v = np.array([2.0, 3.0])
projection = tk.project_onto(u, v)

print(f'dot(u, v)        {tk.dot(u, v):.3f}  (manual {tk.dot_manual(u, v):.3f})')
print(f'angle            {tk.angle_between(u, v, degrees=True):.2f} degrees')
print(f'projection of u  {np.round(projection, 3)}')

fig, ax = plt.subplots(figsize=(5, 5))
for vector, colour, label in [(u, 'steelblue', 'u'), (v, 'seagreen', 'v'), (projection, 'crimson', 'proj_v(u)')]:
    ax.quiver(0, 0, vector[0], vector[1], angles='xy', scale_units='xy', scale=1, color=colour, label=label)
ax.plot([projection[0], u[0]], [projection[1], u[1]], color='crimson', ls=':', lw=1)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.set_title('Projection is orthogonal to the residual')
ax.legend()
plt.show()

In [ ]:
# Least squares = linear regression written as a matrix problem.
x_data = tk.linspace(0, 10, 60)
y_data = 2.5 * x_data + 1.0 + tk.random_normal(60, scale=2.0, seed=SEED)
design = np.column_stack([x_data, np.ones_like(x_data)])
slope, intercept = tk.least_squares(design, y_data)

print(f'fitted  y = {slope:.3f}x + {intercept:.3f}   (true: y = 2.5x + 1)')

fig, ax = plt.subplots()
ax.scatter(x_data, y_data, s=25, color='steelblue', label='noisy observations')
ax.plot(x_data, design @ [slope, intercept], color='crimson', lw=2, label='least_squares fit')
ax.set_title('least_squares on an over-determined system')
ax.legend()
plt.show()

## 5. Simulation — Monte Carlo estimate of pi

Sample points uniformly in the unit square. The fraction landing inside the quarter
circle approaches `pi / 4`, so `pi ~= 4 * inside / n`. Both coordinates are drawn as
one `(n, 2)` array and tested with a single comparison.

In [ ]:
result = tk.monte_carlo_pi(1_000_000, seed=SEED)
print(result)
print(f'relative error: {result.relative_error:.4%}')

# A small sample, plotted, to show what is actually being counted.
points = tk.rng_from(SEED).random((3_000, 2))
inside = np.sum(points**2, axis=1) <= 1.0

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(points[inside, 0], points[inside, 1], s=4, color='steelblue', label='inside')
ax.scatter(points[~inside, 0], points[~inside, 1], s=4, color='lightcoral', label='outside')
arc = tk.linspace(0, np.pi / 2, 200)
ax.plot(np.cos(arc), np.sin(arc), color='black', lw=1.5)
ax.set_aspect('equal')
ax.set_title(f'3,000 points -> pi ~= {4 * inside.mean():.4f}')
ax.legend(loc='lower left')
plt.show()

In [ ]:
# Monte Carlo error shrinks like 1/sqrt(n) — ten times the samples for one more digit.
sizes = np.logspace(2, 6, 25).astype(int)
errors = np.array([np.mean([tk.monte_carlo_pi(n, seed=s).absolute_error for s in range(12)]) for n in sizes])

fig, ax = plt.subplots()
ax.loglog(sizes, errors, 'o-', color='steelblue', label='mean |error| over 12 seeds')
ax.loglog(sizes, errors[0] * np.sqrt(sizes[0] / sizes), 'k--', lw=1, label='1 / sqrt(n) reference')
ax.set_xlabel('samples')
ax.set_ylabel('absolute error')
ax.set_title('Monte Carlo convergence')
ax.legend()
plt.show()

In [ ]:
# Random walks: 300 independent walks built from one cumsum.
steps = 1_000
walks = tk.random_walk(steps=steps, walks=300, seed=SEED)
time_axis = np.arange(steps + 1)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(walks[:60].T, lw=0.7, alpha=0.6)
left.plot(time_axis, np.sqrt(time_axis), 'k--', lw=1.5, label='+/- sqrt(n)')
left.plot(time_axis, -np.sqrt(time_axis), 'k--', lw=1.5)
left.set_title('60 walks and the sqrt(n) envelope')
left.set_xlabel('step')
left.legend()

finals = walks[:, -1]
right.hist(finals, bins=30, color='steelblue', edgecolor='none')
right.set_title(f'final positions: std={finals.std():.1f} (theory {np.sqrt(steps):.1f})')
fig.tight_layout()
plt.show()

In [ ]:
# Bootstrap: resample the data itself to get a confidence interval, no formula needed.
sample = tk.random_normal(200, loc=5.0, scale=2.0, seed=SEED)
low, high = tk.bootstrap_ci(sample, resamples=10_000, confidence=0.95, seed=SEED)
print(f'sample mean       {sample.mean():.4f}')
print(f'95% bootstrap CI  [{low:.4f}, {high:.4f}]   (true mean 5.0)')

indices = tk.rng_from(SEED).integers(0, sample.size, size=(10_000, sample.size))
means = sample[indices].mean(axis=1)

fig, ax = plt.subplots()
ax.hist(means, bins=60, color='steelblue', edgecolor='none')
ax.axvline(low, color='crimson', lw=2, label='2.5th / 97.5th percentile')
ax.axvline(high, color='crimson', lw=2)
ax.axvline(5.0, color='black', ls='--', lw=1.5, label='true mean')
ax.set_title('Bootstrap distribution of the sample mean (10,000 resamples)')
ax.legend()
plt.show()

## 6. Performance — loops vs vectorization

Same arithmetic, two ways of expressing it.

In [ ]:
report = tk.compare_loop_vs_numpy(1_000_000, seed=SEED)
print(report['loop'])
print(report['numpy'])
print(f"speed-up: {report['speedup']:.1f}x")

n = 120
generator = tk.rng_from(SEED)
A = tk.random_normal((n, n), seed=generator)
B = tk.random_normal((n, n), seed=generator)
naive = tk.time_callable(tk.matmul_manual, A, B, label='matmul (triple loop)', repeats=1)
fast = tk.time_callable(tk.matmul, A, B, label='matmul (BLAS)', repeats=3)
print(f'\n{naive}\n{fast}')
print(f'speed-up: {naive.seconds / fast.seconds:.0f}x')

In [ ]:
labels = ['sum of squares\n(1M elements)', f'matrix multiply\n({n}x{n})']
loop_times = [report['loop'].seconds * 1e3, naive.seconds * 1e3]
numpy_times = [report['numpy'].seconds * 1e3, fast.seconds * 1e3]
positions = np.arange(len(labels))

fig, ax = plt.subplots()
ax.bar(positions - 0.2, loop_times, width=0.4, color='lightcoral', label='python loop')
ax.bar(positions + 0.2, numpy_times, width=0.4, color='steelblue', label='numpy')
for position, loop_ms, numpy_ms in zip(positions, loop_times, numpy_times):
    ax.text(position, max(loop_ms, numpy_ms) * 1.4, f'{loop_ms / numpy_ms:.0f}x faster', ha='center')
ax.set_yscale('log')
ax.set_xticks(positions, labels)
ax.set_ylabel('milliseconds (log scale)')
ax.set_title('Vectorization pay-off')
ax.legend()
plt.show()

In [ ]:
# How the gap grows with problem size.
sizes = [1_000, 10_000, 100_000, 1_000_000]
speedups = [tk.compare_loop_vs_numpy(size, seed=SEED)['speedup'] for size in sizes]

fig, ax = plt.subplots()
ax.semilogx(sizes, speedups, 'o-', color='steelblue')
ax.set_xlabel('array size')
ax.set_ylabel('speed-up (loop time / numpy time)')
ax.set_title('The advantage grows as the per-call overhead is amortised')
plt.show()

## 7. Saving and loading arrays

`.npy` holds one array, `.npz` holds a named bundle. Both preserve dtype and shape
exactly — no CSV rounding.

In [ ]:
features = tk.random_normal((1_000, 5), seed=SEED)
labels = tk.random_integers(1_000, 0, 3, seed=SEED)

single = tk.save_array('../data/features', features)
bundle = tk.save_arrays('../data/dataset', features=features, labels=labels)
print(f'{single.name}: {single.stat().st_size:,} bytes')
print(f'{bundle.name}: {bundle.stat().st_size:,} bytes (compressed)')

loaded = tk.load_arrays(bundle)
print('\nkeys           ', sorted(loaded))
print('identical      ', np.array_equal(loaded['features'], features))
print('dtype preserved', loaded['labels'].dtype == labels.dtype)

## Takeaways

- **Array thinking.** Once the data is in an array, the loop disappears into the
  operation — `np.sum(a**2)` instead of accumulating in Python.
- **Broadcasting replaces nested loops.** `pairwise_distances` computes an `(n, m)`
  matrix from two reshapes and one subtraction.
- **Correctness first, speed second.** Every `*_manual` function exists so the
  vectorized one can be tested against it; `pytest tests -q` runs 103 such checks.
- **Randomness is a seeded generator, not global state.** `default_rng(seed)` makes
  every simulation in this notebook exactly reproducible.
- **Errors shrink like `1 / sqrt(n)`.** That single fact governs Monte Carlo, the
  bootstrap, and the standard error of a mean alike.